# TRPO — Trust Region Policy Optimization (2015)

_Papers · Reinforcement Learning_

**Maximize a surrogate of the return, but only within a trust region — keep the new policy's average KL-divergence from the old one below a small budget — and improvement is (near-)monotonic.**

---

This notebook is a practice scaffold for the **TRPO — Trust Region Policy Optimization (2015)** lesson. We build it one piece at a time: the policy and its surrogate, the trust-region line search, then an ablation that removes the KL constraint. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy (Colab / any Python)

### Step 1 — Build the toy policy, its surrogate, and the KL divergence

We work with the simplest possible policy: a single state with a softmax over **three actions**. The advantages `A` are estimated under the *old* policy — action 0 is above average, the other two below. The **surrogate** is the importance-sampled objective, which for this single-state case reduces to the expected advantage `sum_a pi(a) A(a)`; its gradient w.r.t. the logits is `pi * (A - mean_A)`. `kl_old_new` measures how far a candidate policy has drifted from the old one — this is the quantity the trust region will cap.

In [ ]:
import numpy as np

def softmax(z):
    z = z - np.max(z)               # subtract the max for numerical stability
    e = np.exp(z)
    return e / e.sum()

# Single-state policy over 3 actions. Advantages estimated under the OLD policy.
A = np.array([1.0, -0.5, -0.5])     # action 0 is good; the other two below average

def surrogate(z):
    # importance-sampled surrogate reduces to sum_a pi(a) A(a)
    pi = softmax(z)
    return float((pi * A).sum())

def grad_surr(z):
    # d/d(logits) of sum_a pi(a)A(a) = pi * (A - mean_A)
    pi = softmax(z)
    mean_A = (pi * A).sum()
    return pi * (A - mean_A)

def kl_old_new(z_old, z_new):
    # D_KL( pi_old || pi_new )  -- old is the reference (Eq. 12)
    p_old = softmax(z_old)
    p_new = softmax(z_new)
    return float((p_old * np.log(p_old / p_new)).sum())

### Step 2 — Take one constrained trust-region step

Starting from a uniform old policy, we move along the ascent direction `g` but only as far as the **KL budget** `DELTA` allows. `trust_region_beta` bisects to find the largest step size `beta` whose new policy still satisfies `KL(old||new) <= DELTA` — the step lands exactly on the trust-region boundary (Eq. 12). The result: action 0's probability rises, the surrogate improves, and the realized KL is pinned at the budget.

In [ ]:
# --- Worked example: one constrained trust-region step from the uniform old policy. ---
DELTA = 0.02
z_old = np.array([0.0, 0.0, 0.0])           # uniform old policy
g = grad_surr(z_old)                         # ascent direction
print("pi_old =", np.round(softmax(z_old), 4), " surrogate_old =", round(surrogate(z_old), 4))
print("ascent direction g =", np.round(g, 4))

# Line-search the largest beta along g with mean KL <= DELTA  (Eq. 12).
def trust_region_beta(z_old, g, delta, hi=300.0):
    lo = 0.0
    for _ in range(60):                       # bisection to the KL boundary
        mid = 0.5 * (lo + hi)
        z_mid = z_old + mid * g
        if kl_old_new(z_old, z_mid) < delta:
            lo = mid                          # still inside the budget -> push further
        else:
            hi = mid                          # overshot the budget -> pull back
    return lo

beta = trust_region_beta(z_old, g, DELTA)
z_new = z_old + beta * g
print("beta at KL boundary =", round(beta, 4))
print("pi_new =", np.round(softmax(z_new), 4),
      " KL =", round(kl_old_new(z_old, z_new), 4),
      " surrogate_new =", round(surrogate(z_new), 4))
# Expect:  beta ~ 0.83,  pi_new ~ [0.431 0.284 0.284],  KL = 0.02,  surrogate ~ 0.147

### Step 3 — Iterate the step, then ablate the trust region

The surrogate is only a *local* model of the true return: a stale-data penalty grows once the policy moves more than a little, so a too-large step actually **lowers** the realized true return. We run two policies from the same start: the **constrained** TRPO step (largest move inside the KL ball each iteration) and an **unconstrained** ablation (a fixed big greedy step up the same gradient, ignoring KL). The constrained run climbs smoothly toward the optimum; the unconstrained run overshoots on its first step and crashes the true return before recovering — isolating the trust region as the cause of reliable improvement.

In [ ]:
# --- Iterate the step; ABLATE the trust region. ---
# The surrogate is only a LOCAL model: a stale-data penalty grows once the policy moves
# more than a small amount, so a too-large step LOWERS the realized true return.
def realized_true_return(z_old, z_new):
    p_old = softmax(z_old)
    p_new = softmax(z_new)
    move = np.linalg.norm(p_new - p_old)              # how far the policy moved
    surrogate_gain = float((p_new * A).sum())
    stale_penalty = 6.0 * max(0.0, move - 0.18) ** 2  # grows once we move too far
    return surrogate_gain - stale_penalty

def run(constrained, iters=12, delta=0.02, big_lr=14.0):
    z = np.zeros(3)
    traj = []
    for _ in range(iters):
        z_old = z.copy()
        g = grad_surr(z_old)
        if constrained:
            b = trust_region_beta(z_old, g, delta)    # largest step inside the KL ball
            z = z_old + b * g
        else:
            z = z_old + big_lr * g                    # ABLATION: fixed big greedy step, no KL constraint
        traj.append(round(realized_true_return(z_old, z), 3))
    return traj

print("\nconstrained (TRPO, delta=0.02) realized true return:", run(True))
print("unconstrained (no trust region)  realized true return:", run(False))
# Constrained -> climbs smoothly toward the optimum (monotonic improvement).
# Unconstrained -> first step OVERSHOOTS the trust region; realized true return crashes
# below the start (~ -1.4) before recovering. (Our small run, not the paper's numbers.)

## Visualize the data & results

_Does taking the largest surrogate-improving step that stays inside the KL trust region (mean KL <= delta) raise the TRUE return monotonically, and does removing the trust region (a fixed big greedy step up the same surrogate gradient) overshoot and lower it? We iterate both on a toy softmax policy and plot the realized true return per step._

In [ ]:
# Sketch of how the two curves above are produced (full loop in the CODE cell).
# Iterate two updates on a 3-action softmax policy with the same surrogate gradient g:
#
#   constrained:   z <- z + beta*g   with beta the LARGEST step s.t. mean KL(old||new) <= delta  (Eq. 12)
#   unconstrained: z <- z + big_lr*g  (fixed big step, KL ignored)  -- the ABLATION
#
# We plot the realized TRUE return per step (surrogate gain minus a stale-data penalty that
# grows once the policy moves too far -- the surrogate is only a LOCAL model of the return).
#
#   constrained   -> smooth monotone climb to the optimum (Theorem 1's monotonic improvement)
#   unconstrained -> first step overshoots the trust region; true return crashes to ~ -1.4
#
# (Numbers are from our own run; the paper reports MuJoCo locomotion + Atari results with
#  delta=0.01, not these toy values.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The constrained step. Single-state policy over three actions, old policy uniform
            $\pi_{old}=(\tfrac13,\tfrac13,\tfrac13)$, estimated advantages $A=(+1,-0.5,-0.5)$, trust-region
            size $\delta=0.02$. Compute the surrogate's value at the old policy, the ascent direction, and the
            new policy after the largest step with mean KL $\le \delta$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Surrogate at old policy: $\sum_a \pi(a)A(a) = \tfrac13(1)+\tfrac13(-0.5)+\tfrac13(-0.5) = 0$. — _The importance-sampled surrogate reduces to $\sum_a\pi(a)A(a)$; at uniform it is the mean advantage, here $0$._
- Ascent direction (gradient w.r.t. logits): $g = \pi(A-\bar A) = (0.333,-0.167,-0.167)$. — _It points toward the positive-advantage action 0 and away from the others — the direction that raises the surrogate._
- Line-search $\beta$ along $g$ until $D_{KL}(\pi_{old}\|\pi_{new}) = \delta$: boundary at $\beta\approx 0.83$. — _Eq. 12 takes the BIGGEST step inside the KL budget; the largest $\beta$ with KL $\le 0.02$ is on the boundary._
- Read off $\pi_{new} = \text{softmax}(0.277,-0.139,-0.139) = (0.431,0.284,0.284)$, KL $=0.020$, surrogate $=0.147$. — _Action 0's probability rose $0.333\to0.431$; the step stopped exactly at the trust-region boundary._

**Answer:** The surrogate rose from $0$ to $0.147$ and action 0's probability rose from $0.333$ to $0.431$,
                 with the realized KL pinned at the budget $\delta=0.02$. The trust region bound the step:
                 we improved as much as the KL budget allowed and no further. The notebook recomputes
                 $g=(0.333,-0.167,-0.167)$, $\beta\approx0.83$, $\pi_{new}=(0.431,0.284,0.284)$.

</details>

**Problem 2.** The ablation. Your constrained TRPO step raises the true return smoothly toward its
            optimum. Replace the line search with a single fixed big step up the surrogate gradient
            (ignore the KL constraint), keeping the advantages, policy form, and start identical, and iterate.
            What happens to the true-return curve on the FIRST iteration, and what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the step rule: delete the KL line search; use $\theta \leftarrow \theta_{old} + (\text{big})\cdot g$. — _An honest ablation changes exactly one thing — the trust-region constraint — so any difference is attributable to it._
- Iterate and watch the REALIZED true return: the unconstrained first step overshoots the trust region and the true return drops below the starting value (in our run, to about $-1.4$), while the constrained run climbed to $\approx 0.15$. — _The surrogate is only a local model; far outside the KL ball it over-credits the move, so a big step that "improves the surrogate" actually lowers the true return._
- Conclude the KL constraint, not the gradient direction, is what makes the improvement reliable. — _Both runs ascend the same surrogate gradient; only the constrained one improves the true return monotonically, isolating the trust region as the cause._

**Answer:** The unconstrained agent overshoots on the very first step &mdash; its realized true return
                 crashes below where it started (about $-1.4$ in our run) because it trusted the surrogate
                 outside its valid neighborhood &mdash; while the constrained TRPO step climbs smoothly toward the
                 optimum. Since the only difference is the mean-KL $\le\delta$ constraint (Eq. 12), this isolates
                 the trust region as the source of monotonic improvement. The CODEVIZ panel shows this
                 contrast.

</details>

**Problem 3.** TRPO vs PPO. TRPO enforces the trust region with a hard constraint $\bar D_{KL}\le\delta$
            solved by conjugate gradient + line search. How does PPO (paper-ppo) get a similar
            "don't move too far" effect more cheaply, and what does it give up?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall PPO's objective: it CLIPS the probability ratio $r_t$ into $[1-\epsilon,1+\epsilon]$ inside the loss and takes a pessimistic minimum (PPO Eq. 7). — _Clipping removes the incentive to push any single action's ratio far, approximating a trust region per-sample._
- Note PPO needs only first-order SGD and allows several epochs of minibatch reuse per batch. — _No constrained second-order solve, no Fisher-vector products, no line search — far simpler and compatible with shared policy/value nets._
- Identify what is given up: PPO bounds per-sample drift, not the TOTAL KL, so it loses TRPO's explicit monotonic-improvement guarantee. — _The clip is a heuristic surrogate for the trust region; enough epochs can still let the policy drift, which TRPO's hard KL constraint forbids._

**Answer:** PPO replaces TRPO's hard KL constraint with a clipped surrogate: it clamps the
                 importance-sampling ratio into $[1-\epsilon,1+\epsilon]$ and takes a pessimistic min, which
                 approximates the trust region with cheap first-order SGD and lets you reuse each batch for several
                 epochs. It gives up TRPO's exact average-KL constraint and its monotonic-improvement guarantee in
                 exchange for simplicity and speed &mdash; which is why PPO became the practical default. See
                 paper-ppo.

</details>